# Feature Engineering: Crop Recommendation Model
**Agri-DSS Practicum, CMU Africa Fall 2026**  
Authors: Arsene (Part A, data ingestion) | Thierry (Part B, modelling)  
Last updated: 2026-10-26

This notebook documents every data join and modelling decision made during Part A. It is the authoritative reference for how the feature table was built and why each choice was made.

**Four pilot districts:** Nyagatare, Musanze, Nyabihu, Ruhango  
**Target variable:** crop type (maize, beans, rice, Irish potato, and intercrop classes)  
**Base table:** CIAT fieldwork points, 2,739 observations, collected May-June 2025

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# All paths relative to the repo root (agri-dss/)
CIAT_POINTS   = 'data/raw/ciat/ciat_points_clean.csv'
NASA_POWER    = 'data/raw/nasa_power/nasa_power_seasonal.csv'
NISR_YIELD    = 'data/raw/nisr/nisr_yield_kg_ha.csv'
NISR_PROD     = 'data/raw/nisr/nisr_production_mt.csv'
NISR_HARV     = 'data/raw/nisr/nisr_harvested_area_ha.csv'
NISR_CULT     = 'data/raw/nisr/nisr_cultivated_area_ha.csv'
ISDA_POINTS   = 'data/raw/isda/isda_soil_points.csv'  # push pending if run not complete

print('Libraries loaded.')

---
## 1. Data Sources

| Source | What it provides | Join level | Rows |
|---|---|---|---|
| CIAT Fieldwork Points | Crop labels, GPS coordinates, elevation | Base table | 2,739 |
| NASA POWER | Seasonal climate (rainfall, temp, solar) | District + season + year | 384 |
| NISR SAS | Historical crop statistics | District + season + year + crop | ~12,800 |
| iSDAsoil REST API | Soil properties at 30m resolution | Point level (lat/lon) | 2,739 |

**Excluded sources:**
- Rwanda Govt Soil Fertility dataset (148 samples): all in Gicumbi district, none in pilot districts.
- CIMMYT 2022 soil dataset (104 samples): coordinates truncated for privacy, too sparse to join spatially.

---
## 2. CIAT Base Table

CIAT collected 2,739 georeferenced crop observations across the four pilot districts between May 6 and June 7, 2025. This is the base table for all joins.

The shapefile includes `GPS_Elev` (elevation in metres) with 100% coverage, so no external elevation API was needed.

In [ ]:
ciat = pd.read_csv(CIAT_POINTS)
print('Shape:', ciat.shape)
print()
print('Points per district:')
print(ciat['District'].value_counts())
print()
print('Crop labels (landcover):')
print(ciat['landcover'].value_counts(dropna=False))

In [ ]:
print('Coordinate bounds:')
print('  Latitude : {:.4f} to {:.4f}'.format(ciat['latitude'].min(), ciat['latitude'].max()))
print('  Longitude: {:.4f} to {:.4f}'.format(ciat['longitude'].min(), ciat['longitude'].max()))
print('  Elevation: {:.1f} to {:.1f} m'.format(ciat['GPS_Elev'].min(), ciat['GPS_Elev'].max()))
print()
print('Elevation == 0.0 (GPS errors):',  (ciat['GPS_Elev'] == 0.0).sum())
print('Longitude > 31.0 (outside Rwanda):', (ciat['longitude'] > 31.0).sum())

### Data quality filters

Two coordinate issues were found in the raw CIAT data:

1. **Elevation = 0.0 m**: Rwanda's lowest point is around 950m above sea level. Any GPS_Elev reading of 0.0 is a GPS acquisition failure, not a real observation. Filter: `elevation_m > 900`.

2. **Longitude > 31.0**: Rwanda's eastern boundary is approximately 30.9° E. Points beyond this are outside the country. Filter: `longitude < 31.0`.

These filters should be applied before training, not during ingestion, so the full raw dataset is preserved.

In [ ]:
# Assign season and year to CIAT points
# CIAT collected May-June 2025 = Rwanda Season B
# NASA POWER uses start-year convention: Season B start year 2024 covers Mar 2025 to Jun 2025
# This is the key alignment: join on year=2024, not year=2025
ciat['season'] = 'B'
ciat['year'] = 2024
print('Season and year assigned: season=B, year=2024')
print('This matches NASA POWER Season B start year 2024 (Mar 2025 - Jun 2025)')

### Season/year alignment decision

Rwanda has three growing seasons:
- Season A: September 1 to January 31 of the following year
- Season B: March 1 to June 30
- Season C: June 1 to August 31

CIAT collected data in May-June 2025, which falls in Season B. NASA POWER uses a **start-year convention**: Season B labelled as year 2024 covers March 2025 to June 2025. Joining on `year=2025` returns all nulls. Joining on `year=2024` gives 0 nulls across all 2,739 rows.

---
## 3. Climate Features (NASA POWER)

NASA POWER provides daily satellite-derived climate data. We fetched four variables for each pilot district, aggregated by season (2017-2024):
- `PRECTOTCORR`: precipitation (mm), seasonal sum
- `T2M_MAX`: maximum 2m air temperature (°C), seasonal mean
- `T2M_MIN`: minimum 2m air temperature (°C), seasonal mean
- `ALLSKY_SFC_SW_DWN`: surface solar radiation (MJ/m²), seasonal mean

The raw file is in long format (one row per district-season-year-variable). It must be pivoted to wide format before joining.

In [ ]:
climate = pd.read_csv(NASA_POWER)
print('Raw NASA POWER shape (long):', climate.shape)
print('Variables:', sorted(climate['variable'].unique()))
print()
print(climate.head(8).to_string())

In [ ]:
# Pivot long to wide
climate_wide = climate.pivot_table(
    index=['district', 'season', 'year'],
    columns='variable',
    values='value'
).reset_index()
climate_wide.columns.name = None

climate_wide = climate_wide.rename(columns={
    'PRECTOTCORR':       'precip_mm',
    'T2M_MAX':           'tmax_c',
    'T2M_MIN':           'tmin_c',
    'ALLSKY_SFC_SW_DWN': 'solar_mj_m2'
})

print('Wide shape:', climate_wide.shape)
print()
# Show Season B 2024 for the four pilot districts
mask = (climate_wide['season'] == 'B') & (climate_wide['year'] == 2024)
print('Season B 2024 (the join target):')
print(climate_wide[mask].to_string(index=False))

In [ ]:
# Join climate to CIAT
merged = ciat.merge(
    climate_wide,
    left_on=['District', 'season', 'year'],
    right_on=['district', 'season', 'year'],
    how='left'
).drop(columns=['district'])

climate_cols = ['precip_mm', 'tmax_c', 'tmin_c', 'solar_mj_m2']
print('After climate join:')
print('  Shape:', merged.shape)
print('  Nulls in climate columns:')
print(merged[climate_cols].isnull().sum())
print()
print('Sample (one row per district):')
print(merged.groupby('District').first()[climate_cols].to_string())

---
## 4. Historical Crop Statistics (NISR)

NISR SAS (Seasonal Agricultural Survey) provides district-level crop statistics for each season and year: cultivated area, harvested area, production, and yield.

### Crop name mapping decision

CIAT uses plot-level landcover labels, including intercrop combinations (e.g. "Bean and Maize"). NISR reports single crops only. The mapping below was chosen as follows:

- **Pure crops** (Maize, Bean, Rice, Irish potato): map directly to the NISR equivalent.
- **Intercrop combinations** (Bean and Maize, Irish Potato and Maize, etc.): leave NISR columns as null. These points are kept in the dataset so Thierry can treat intercrop labels as separate target classes during modelling, or drop them if predicting only pure monocrops.

This is more honest than picking an arbitrary primary crop for intercrop plots, and it preserves optionality for the modelling step.

In [ ]:
# Load all four NISR tables
nisr_yield = pd.read_csv(NISR_YIELD).rename(columns={'value': 'nisr_yield_kg_ha'})
nisr_prod  = pd.read_csv(NISR_PROD).rename(columns={'value': 'nisr_production_mt'})
nisr_harv  = pd.read_csv(NISR_HARV).rename(columns={'value': 'nisr_harvested_area_ha'})
nisr_cult  = pd.read_csv(NISR_CULT).rename(columns={'value': 'nisr_cultivated_area_ha'})

# Merge all four into one NISR table
nisr = nisr_yield
nisr = nisr.merge(nisr_prod, on=['year','season','district','crop'], how='outer')
nisr = nisr.merge(nisr_harv, on=['year','season','district','crop'], how='outer')
nisr = nisr.merge(nisr_cult, on=['year','season','district','crop'], how='outer')

print('NISR combined shape:', nisr.shape)
print('Seasons in NISR:', sorted(nisr['season'].unique()))
print('Year range:', nisr['year'].min(), 'to', nisr['year'].max())

In [ ]:
# Crop name mapping: pure crops only; intercrop -> None (leaves NISR columns null)
crop_map = {
    'Maize':                  'Maize',
    'Bean':                   'Beans',
    'Rice':                   'Paddy rice',
    'Irish potato':           'Irish potato',
    'Irish Potato':           'Irish potato',
    # Intercrop combinations: no NISR mapping
    'Bean and Maize':         None,
    'Maize and Bean':         None,
    'Bean and Irish potato':  None,
    'Irish Potato and Maize': None,
    'Irish potato and Maize': None,
    'Cassava and Maize':      None,
}

merged['nisr_crop'] = merged['landcover'].map(crop_map)

print('CIAT landcover -> nisr_crop mapping counts:')
print(merged.groupby('landcover')['nisr_crop'].first().to_string())
print()
print('Points with a NISR crop mapping:', merged['nisr_crop'].notna().sum())
print('Points without (intercrop/null):  ', merged['nisr_crop'].isna().sum())

In [ ]:
# Join NISR to merged table
merged = merged.merge(
    nisr,
    left_on=['District', 'season', 'year', 'nisr_crop'],
    right_on=['district', 'season', 'year', 'crop'],
    how='left'
).drop(columns=['district', 'crop'])

nisr_cols = ['nisr_yield_kg_ha', 'nisr_production_mt', 'nisr_harvested_area_ha', 'nisr_cultivated_area_ha']
print('After NISR join:')
print('  Shape:', merged.shape)
print('  Nulls in NISR columns (expected = intercrop count):')
print(merged[nisr_cols].isnull().sum())
print()
print('Sample with NISR values:')
sample = merged[merged['nisr_yield_kg_ha'].notna()][['District','landcover','nisr_crop','nisr_yield_kg_ha','nisr_production_mt']]
print(sample.head(6).to_string(index=False))

---
## 5. Soil Features (iSDAsoil)

iSDAsoil provides modelled soil properties at 30m resolution across Africa via a REST API. We queried 9 properties at 2 depths (0-20cm and 20-50cm) for each of the 2,739 CIAT points.

Properties: nitrogen, phosphorus, potassium, pH, sand, clay, silt, organic carbon, cation exchange capacity.

The full run takes several hours. If `isda_soil_points.csv` does not exist yet, skip this section and re-run after the overnight job completes.

**Why iSDAsoil instead of lab-measured data:**
- Rwanda Govt soil dataset: 148 samples, all in Gicumbi district (not a pilot district).
- CIMMYT 2022: 104 MIRS-predicted samples with truncated coordinates, cannot join spatially.
- iSDAsoil covers all four pilot districts at every CIAT point location.

In [ ]:
import os

if not os.path.exists(ISDA_POINTS):
    print('iSDAsoil file not found. Run isda_soil_fetcher.py and re-run this cell.')
    isda_available = False
else:
    isda = pd.read_csv(ISDA_POINTS)
    print('iSDAsoil shape:', isda.shape)
    print('Columns:', list(isda.columns))
    print()
    print('Null counts:')
    print(isda.isnull().sum()[isda.isnull().sum() > 0])
    isda_available = True

In [ ]:
if isda_available:
    # Join iSDAsoil at point level using F_index as the key
    # F_index in CIAT matches point_id in iSDAsoil output
    isda_cols = [c for c in isda.columns if c != 'point_id']
    merged = merged.merge(
        isda,
        left_on='F_index',
        right_on='point_id',
        how='left'
    ).drop(columns=['point_id'])

    print('After iSDAsoil join:')
    print('  Shape:', merged.shape)
    soil_cols = [c for c in merged.columns if any(c.startswith(p) for p in ['nitrogen','phosphorous','potassium','ph_','sand_','clay_','silt_','carbon_organic','cation_'])]
    print('  Nulls in soil columns:')
    print(merged[soil_cols].isnull().sum())
else:
    print('Skipping iSDAsoil join. Re-run after isda_soil_fetcher.py completes.')

---
## 6. Final Feature Table

Summary of what has been joined and what remains.

In [ ]:
print('Final feature table shape:', merged.shape)
print()
print('Column list:')
for c in merged.columns:
    print(' ', c)
print()
print('Overall null counts (top 20):')
nulls = merged.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False).head(20))

In [ ]:
# Target variable distribution
print('Target variable (landcover) distribution:')
print(merged['landcover'].value_counts(dropna=False))
print()
print('Total labelled points:', merged['landcover'].notna().sum())
print('Unlabelled points:    ', merged['landcover'].isna().sum())

### Pre-processing checklist for Thierry

Before training, apply these filters and transformations:

1. `merged = merged[merged['GPS_Elev'] > 900]` - removes GPS elevation errors (0.0m readings)
2. `merged = merged[merged['longitude'] < 31.0]` - removes coordinate outliers outside Rwanda
3. Drop columns with more than 30% missing values
4. One-hot encode `district` and `season` for tree models, or leave as-is for embedding-based models
5. Intercrop points (null in `nisr_yield_kg_ha`) can be kept as their own target classes or dropped, depending on the modelling approach

Refer to `docs/feature_schema.md` in the repo for full column definitions and units.

In [ ]:
# Save the feature table (without iSDAsoil if not yet available)
import os
os.makedirs('data/processed', exist_ok=True)
out_path = 'data/processed/feature_table.csv'
merged.to_csv(out_path, index=False)
print('Saved:', out_path)
print('Shape:', merged.shape)